In [8]:
import os
from dotenv import load_dotenv
from FinMind.data import DataLoader
import pandas as pd
import json
import numpy as np
from hmmlearn.hmm import GaussianHMM
from tqdm import tqdm
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import confusion_matrix

## Hmmlearn

### Utility functions

In [9]:
def print_matrix(name, mat, precision=3):
    """Pretty-print a matrix or vector."""
    np.set_printoptions(precision=precision, suppress=True)
    print(f"\n{name}:\n{mat}")


def align_state_labels(y_true, y_pred, n_states):
    """
    Align predicted HMM state labels to true labels using the Hungarian algorithm.

    Why needed:
    HMM state IDs are arbitrary. The model may call the true state 0 as state 2, etc.
    This function finds the best one-to-one mapping between predicted and true labels.
    """
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(n_states))
    # Hungarian solves a minimum-cost assignment problem,
    # so we negate confusion counts to maximize matches.
    row_ind, col_ind = linear_sum_assignment(-cm)

    mapping = {}
    for true_label, pred_label in zip(row_ind, col_ind):
        mapping[pred_label] = true_label

    aligned = np.array([mapping[p] for p in y_pred], dtype=int)
    return aligned, cm, mapping


def compute_state_accuracy(y_true, y_pred):
    """Simple classification accuracy."""
    return np.mean(y_true == y_pred)

### Part 1. Generate synthetic data from a known Gaussian HMM

In [ ]:
# Reproducibility
seed = 42
rng = np.random.default_rng(seed)
np.random.seed(seed)  # hmmlearn uses NumPy's global random state internally

n_states = 3
n_features = 4
n_samples = 1000

# Ground-truth HMM parameters
startprob_true = np.array([0.60, 0.30, 0.10])

transmat_true = np.array([
    [0.85, 0.10, 0.05],
    [0.08, 0.82, 0.10],
    [0.06, 0.14, 0.80],
])

means_true = np.array([
    [0.0,  0.0,  0.0,  0.0],
    [4.0,  4.0, -3.0,  2.0],
    [-4.0, 3.0,  3.0, -2.0],
])

# Full covariance matrices: each state has a 4x4 covariance matrix
covars_true = np.array([
    [
        [1.0,  0.3,  0.2,  0.0],
        [0.3,  1.2,  0.1, -0.2],
        [0.2,  0.1,  0.9,  0.25],
        [0.0, -0.2,  0.25, 1.1],
    ],
    [
        [1.5,  0.4, -0.1,  0.2],
        [0.4,  1.3,  0.3,  0.1],
        [-0.1, 0.3,  1.1, -0.15],
        [0.2,  0.1, -0.15, 1.4],
    ],
    [
        [1.2, -0.35, 0.25,  0.1],
        [-0.35, 1.6, 0.2, -0.05],
        [0.25, 0.2,  1.0,  0.3],
        [0.1, -0.05, 0.3,  1.2],
    ],
])

# Build the "true" model only for sampling
true_model = GaussianHMM(
    n_components=n_states,
    covariance_type="full",
    random_state=seed
)
true_model.startprob_ = startprob_true
true_model.transmat_ = transmat_true
true_model.means_ = means_true
true_model.covars_ = covars_true

# Generate observations X and hidden states Z_true
X, Z_true = true_model.sample(n_samples)

print("=== Part 1: Synthetic data generation ===")
print(f"Observation shape: {X.shape}")          # (n_samples, 4)
print(f"Hidden-state shape: {Z_true.shape}")    # (n_samples,)
print(f"First 10 hidden states: {Z_true[:10]}")
print(f"First 3 observations:\n{X[:3]}")

print_matrix("True start probabilities", startprob_true)
print_matrix("True transition matrix", transmat_true)
print_matrix("True means", means_true)


=== Part 1: Synthetic data generation ===
Observation shape: (1000, 4)
Hidden-state shape: (1000,)
First 10 hidden states: [0 0 0 0 0 0 0 0 2 2]
First 3 observations:
[[ 1.272  0.998 -0.165  0.252]
 [ 0.079  0.576  0.78   0.265]
 [ 2.163  1.769  2.217 -0.839]]

True start probabilities:
[0.6 0.3 0.1]

True transition matrix:
[[0.85 0.1  0.05]
 [0.08 0.82 0.1 ]
 [0.06 0.14 0.8 ]]

True means:
[[ 0.  0.  0.  0.]
 [ 4.  4. -3.  2.]
 [-4.  3.  3. -2.]]

=== Part 2: Learning HMM parameters from observations ===


Training HMM: 100%|██████████| 1000/1000 [00:04<00:00, 243.95it/s]


Training finished.



### Part 2. Learn a new Gaussian HMM from observations only with a progress bar

In [ ]:
# We will fit one EM iteration at a time so we can display a progress bar.
# First iteration: initialize parameters automatically.
# Later iterations: keep the learned parameters and continue updating.

max_em_iterations = 1000
scores = []

learned_model = GaussianHMM(
    n_components=n_states,
    covariance_type="full",
    n_iter=1,               # one EM step per fit call
    init_params="stmc",     # initialize startprob/transmat/means/covars only once
    params="stmc",          # then keep updating them
    random_state=123
)

for i in tqdm(range(max_em_iterations), desc="Training HMM"):
    learned_model.fit(X)

    # After the first iteration, do NOT reinitialize parameters again
    learned_model.init_params = ""

    # Track log-likelihood
    log_likelihood = learned_model.score(X)
    scores.append(log_likelihood)

print("\nTraining finished.")

# Decode most likely state sequence
Z_pred = learned_model.predict(X)

### Part 3. Evaluate prediction accuracy

In [15]:
print("\n=== Part 3: Evaluation ===")

# Raw accuracy is usually misleading because HMM labels are permuted arbitrarily
raw_accuracy = compute_state_accuracy(Z_true, Z_pred)

# Align predicted labels to true labels
Z_pred_aligned, conf_before, mapping = align_state_labels(Z_true, Z_pred, n_states)
aligned_accuracy = compute_state_accuracy(Z_true, Z_pred_aligned)

print_matrix("Confusion matrix before alignment (rows=true, cols=pred)", conf_before)
print(f"\nState mapping applied (predicted -> true): {mapping}")
print(f"Raw state accuracy:     {raw_accuracy:.4f}")
print(f"Aligned state accuracy: {aligned_accuracy:.4f}")

# Compare learned parameters against ground truth
print_matrix("Learned start probabilities", learned_model.startprob_)
print_matrix("Learned transition matrix", learned_model.transmat_)
print_matrix("Learned means", learned_model.means_)

print("\nFinal log-likelihood:", scores[-1])
print("First 5 log-likelihood values:", [round(s, 3) for s in scores[:5]])
print("Last 5 log-likelihood values:", [round(s, 3) for s in scores[-5:]])

# Optional extra diagnostics
print("\nState proportions:")
for s in range(n_states):
    true_prop = np.mean(Z_true == s)
    pred_prop = np.mean(Z_pred_aligned == s)
    print(f"State {s}: true={true_prop:.3f}, predicted={pred_prop:.3f}")


=== Part 3: Evaluation ===

Confusion matrix before alignment (rows=true, cols=pred):
[[  0 322   0]
 [417   0   0]
 [  0   0 261]]

State mapping applied (predicted -> true): {np.int64(1): np.int64(0), np.int64(0): np.int64(1), np.int64(2): np.int64(2)}
Raw state accuracy:     0.2610
Aligned state accuracy: 1.0000

Learned start probabilities:
[0. 1. 0.]

Learned transition matrix:
[[0.82  0.094 0.086]
 [0.09  0.851 0.059]
 [0.177 0.031 0.792]]

Learned means:
[[ 4.145  4.047 -3.079  2.08 ]
 [-0.011  0.005 -0.014 -0.014]
 [-4.005  3.014  3.    -2.071]]

Final log-likelihood: -6436.857753005265
First 5 log-likelihood values: [-7418.385, -7015.241, -6925.853, -6719.083, -6451.852]
Last 5 log-likelihood values: [-6436.858, -6436.858, -6436.858, -6436.858, -6436.858]

State proportions:
State 0: true=0.322, predicted=0.322
State 1: true=0.417, predicted=0.417
State 2: true=0.261, predicted=0.261


## FINMIND API

In [43]:
stock_set = set()
for trader_id in [1360, 1400, 1440, 1470, 1480, 1560, 1650, 1660, 7030, 8440, 8960]:
    df = pd.read_parquet(f"../data/brokers/{trader_id}/2021-06-30_to_2026-02-11.parquet")
    # df[df["stock_id"] == "0050"]
    unique_values = set(df["stock_id"].unique())
    stock_set |= unique_values

In [48]:
with open("stock_ids.json", "w") as f:
    json.dump(list(stock_set), f, indent=4)

In [51]:
df = pd.read_parquet("../data/stocks/00886_2021-06-30_to_2026-02-11.parquet")
df

,date,stock_id,Trading_Volume,Trading_money,open,max,min,close,spread,Trading_turnover
0,2021-06-30,00886,333976,7508838,22.46,22.51,22.46,22.50,0.20,77
1,2021-07-01,00886,270813,6068721,22.42,22.45,22.38,22.39,-0.11,76
2,2021-07-02,00886,186220,4170415,22.40,22.41,22.35,22.39,0.00,53
3,2021-07-05,00886,156618,3535870,22.63,22.63,22.55,22.55,0.16,65
4,2021-07-06,00886,147850,3336956,22.55,22.60,22.54,22.57,0.02,51
...,...,...,...,...,...,...,...,...,...,...
1123,2026-02-05,00886,15163,545764,36.02,36.02,35.95,36.02,-0.46,23
1124,2026-02-06,00886,57555,2021871,35.66,35.98,34.46,34.91,-1.11,69
1125,2026-02-09,00886,12529,446541,35.62,35.62,35.61,35.61,0.70,23
1126,2026-02-10,00886,4963,176388,35.61,35.61,35.50,35.50,-0.11,19


In [ ]:
class BrokerTradeTracker:

    def __init__(self, securities_trader: str, securities_trader_id: str, stock_id: str) -> None:
        self.securities_trader: str = securities_trader
        self.securities_trader_id: str = securities_trader_id
        self.stock_id = stock_id

    def add_record(self, df: pd.DataFrame) -> None:
        self.date = df["date"]

In [3]:
import os
from dotenv import load_dotenv
import pandas as pd
from FinMind.data import DataLoader

# 載入 API key
load_dotenv()
token = os.environ["FINMIND_API_KEY"]

# 初始化 API
api = DataLoader()
api.login_by_token(api_token=token)

# 取得 1470 在 2026-03-02 的交易資料
df = api.taiwan_stock_trading_daily_report(
    securities_trader_id="1440",
    date="2026-03-02",
)

# 篩選華邦電 2344
df_2344 = df[df["stock_id"] == "2344"]

# 計算進出金額
if not df_2344.empty:
    df_2344["buy_amount"] = df_2344["buy"] * df_2344["price"]
    df_2344["sell_amount"] = df_2344["sell"] * df_2344["price"]
    df_2344["net_buy"] = df_2344["buy"] - df_2344["sell"]
    df_2344["net_buy_amount"] = df_2344["buy_amount"] - df_2344["sell_amount"]

print(df_2344)

2026-03-02 22:22:37.736 | INFO     | FinMind.data.finmind_api:login_by_token:84 - Login success
2026-03-02 22:22:37.795 | INFO     | FinMind.data.finmind_api:login_by_token:84 - Login success
2026-03-02 22:22:37.796 | INFO     | FinMind.data.finmind_api:get_data:153 - download Dataset.TaiwanStockInfo, data_id: 
2026-03-02 22:22:38.201 | INFO     | FinMind.data.finmind_api:get_data:153 - download Dataset.TaiwanStockPrice, data_id: 
2026-03-02 22:22:40.866 | INFO     | FinMind.data.finmind_api:get_data:153 - download Dataset.TaiwanStockTradingDailyReport, data_id: 


     securities_trader  price      buy    sell securities_trader_id stock_id  \
2348                美林  115.0     1000   80000                 1440     2344   
2349                美林  115.5        0  230000                 1440     2344   
2350                美林  116.0        0   51000                 1440     2344   
2351                美林  117.0        0   89000                 1440     2344   
2352                美林  117.5        0  275600                 1440     2344   
2353                美林  118.0        0   97000                 1440     2344   
2354                美林  119.5        0  137000                 1440     2344   
2355                美林  120.0   152000    1000                 1440     2344   
2356                美林  120.5   619000       0                 1440     2344   
2357                美林  121.0    93000       0                 1440     2344   
2358                美林  121.5    67000   74000                 1440     2344   
2359                美林  122.0    38000  

In [6]:
df_3037 = df[df["stock_id"] == "3037"]


In [ ]:
df_3037.to_csv("test.csv", index=False, encoding="utf-8-sig")

: 

In [9]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)
print(df_3037)

     securities_trader  price    buy   sell securities_trader_id stock_id        date
5130                美林  463.5  77000      0                 1440     3037  2026-03-02
5131                美林  467.0   1000      0                 1440     3037  2026-03-02
5132                美林  467.5   3000      0                 1440     3037  2026-03-02
5133                美林  469.0   4000      0                 1440     3037  2026-03-02
5134                美林  469.5   1000      0                 1440     3037  2026-03-02
...                ...    ...    ...    ...                  ...      ...         ...
5191                美林  499.5  22000      0                 1440     3037  2026-03-02
5192                美林  500.0  14000  54000                 1440     3037  2026-03-02
5193                美林  501.0  20000   1000                 1440     3037  2026-03-02
5194                美林  502.0   3000      0                 1440     3037  2026-03-02
5195                美林  503.0   8000      0           